### VectorDB, Hugging Face & Ollama Part 1 (Q8 to Q10)

In [5]:
# Install Required Libraries
!pip install langchain==0.1.20 langchain-community
!pip install chromadb sentence-transformers pypdf

  Using cached packaging-23.2-py3-none-any.whl.metadata (3.2 kB)
Using cached packaging-23.2-py3-none-any.whl (53 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
build 1.4.0 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.1.53 which is incompatible.
google-cloud-bigquery 3.40.1 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
wheel 0.46.3 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packagin

  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
Using cached packaging-26.0-py3-none-any.whl (74 kB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


In [1]:
# Install zstd
!apt-get update
!apt-get install -y zstd

# Then Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [2]:
# Start Ollama Server
import subprocess
import time

subprocess.Popen(["ollama", "serve"])

# wait for server to start
time.sleep(5)

In [3]:
# Download Models From Ollama
!ollama pull llama2

!ollama pull llama3
# !ollama pull llama3:8b-instruct-q4_K_M

8)  Create a custom LLM using Ollama and Llama2, and run it locally for basic
text prompts.

->

In [4]:
with open("Modelfile", "w") as f:
    f.write("FROM llama2\n")
    f.write("SYSTEM You are an AI tutor that explains Artificial Intelligence and Machine Learning concepts clearly with examples.\n")
    f.write("PARAMETER temperature 0.7\n")
    f.write("PARAMETER top_p 0.9\n")

In [5]:
!cat Modelfile

FROM llama2
SYSTEM You are an AI tutor that explains Artificial Intelligence and Machine Learning concepts clearly with examples.
PARAMETER temperature 0.7
PARAMETER top_p 0.9


In [6]:
!ollama create ai-tutor -f Modelfile

In [7]:
!ollama list

NAME               ID              SIZE      MODIFIED      
ai-tutor:latest    97293d1d568d    3.8 GB    2 seconds ago    
llama2:latest      78e26419b446    3.8 GB    9 seconds ago    
llama3:latest      365c0bd3c000    4.7 GB    8 seconds ago    


In [8]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama2",
        "prompt": "Explain reinforcement learning with an example.",
        "stream": False
    }
)

print(response.json()["response"])


Reinforcement learning is a subfield of machine learning that involves training an agent to take actions in an environment in order to maximize a reward. The agent learns by trial and error, and the goal is to learn the optimal policy that maximizes the cumulative reward over time.

Here's an example to illustrate the concept of reinforcement learning:

Imagine you are training a robot to play a game of soccer. The robot is equipped with sensors to detect the position of the ball and its own position on the field. The robot's actions are to move its body in order to kick the ball and score points. The reward is the number of points the robot scores.

The goal of the robot is to learn the optimal policy to move its body in order to maximize the number of points it scores. The robot learns through trial and error, and the algorithm adjusts the robot's policy based on the feedback from the environment.

Here's how the algorithm works:

1. The robot takes an action (e.g. moves its body in

In [9]:
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA

9) Implement a basic RAG (Retrieval-Augmented Generation) system using
Ollama with Llama3.

->

In [10]:
# Prepare Documents
documents = [
    "Artificial Intelligence is the simulation of human intelligence in machines.",
    "Machine Learning allows computers to learn from data.",
    "Deep Learning uses neural networks with multiple layers."
]

In [11]:
# Embeddings + VectorDB
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = Chroma.from_texts(documents, embedding)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# RAG Query
llm = Ollama(model="llama3")

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever()
)

query = "What is machine learning?"

print(qa.run(query))

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


According to the given context, Machine Learning allows computers to learn from data.


10) A health-tech startup wants to build a chatbot that can answer user
queries based on medical research articles. Propose and explain a solution using Hugging Face models for understanding, VectorDB for retrieval, and Ollama for generation.

->

Proposed Solution

A health-tech startup can build an intelligent medical chatbot using a Retrieval-Augmented Generation (RAG) architecture. This approach combines Hugging Face models for text understanding, a Vector Database for efficient retrieval of relevant research content, and Ollama running a local LLM for response generation.

The system allows the chatbot to answer questions based on actual medical research papers instead of relying only on the language model’s internal knowledge. This improves accuracy, transparency, and reliability, which are crucial in healthcare applications.


System Architecture

```
Medical Research Articles (PDFs)
            ↓
Document Loader & Preprocessing
            ↓
Hugging Face Embedding Model
(sentence-transformers)
            ↓
Vector Database (ChromaDB)
            ↓
Semantic Retrieval
            ↓
Ollama LLM (Llama3)
            ↓
Generated Medical Answer
```


Step-by-Step Explanation

1. Data Collection (Medical Research Articles)

The chatbot is trained using medical research papers in PDF format, such as studies on COVID-19 diagnosis, imaging techniques, or disease symptoms.

These research papers act as the **knowledge base** for the chatbot.

Example paper topic used:

* Chest radiography for COVID-19 detection.


2. Understanding the Text using Hugging Face Models

Hugging Face Sentence Transformers** are used to convert text into **vector embeddings.

Example model: sentence-transformers/all-MiniLM-L6-v2


This model converts each paragraph of the research paper into a numerical vector representation that captures its semantic meaning.

This allows the system to understand context and similarity between queries and documents.

3. Storing Embeddings in a Vector Database

The embeddings generated from the research articles are stored in a Vector Database such as ChromaDB.

The VectorDB enables semantic search, meaning it retrieves information based on meaning rather than exact keyword matches.

For example:

User query: How is chest radiography used to detect COVID-19?

The VectorDB finds document sections discussing radiography imaging and COVID-19 screening.

4. Retrieval of Relevant Information

When a user asks a question:

1. The query is converted into an **embedding vector** using the same Hugging Face model.

2. The VectorDB compares this query vector with stored document embeddings.

3. The system retrieves the **most relevant document chunks**.

This ensures the chatbot retrieves accurate medical information directly from research papers.

5. Answer Generation using Ollama

The retrieved research text is then passed to a local large language model running through Ollama, such as: Llama3


Ollama generates a natural language answer using the retrieved research context.

This step converts technical research information into a clear and understandable response for users.

Example Query

User question: How is chest radiography used to detect COVID-19?

Retrieved research context explains that chest radiography imaging can be used as an alternative screening method to identify lung abnormalities associated with COVID-19 infection.

Generated chatbot response:

> According to the provided context, chest radiography is used to detect COVID-19 by conducting chest radiography imaging (e.g., chest X-ray (CXR) or computed tomography (CT) imaging) and analyzing the images by radiologists to look for visual indicators associated with SARS-CoV-2 viral infection.

Advantages of This Approach

1. Accurate answers from research papers
   The chatbot retrieves real medical information instead of guessing.

2. Reduced hallucination
   The model relies on retrieved evidence before generating responses.

3. Scalable knowledge base
   New medical research papers can easily be added to the system.

4. Privacy and local inference
   Using Ollama allows the system to run locally without sending sensitive medical data to external APIs.

In [13]:
# Download Multiple Medical Research Papers

!wget https://arxiv.org/pdf/2003.09871.pdf -O covid_paper.pdf
!wget https://arxiv.org/pdf/2005.00001.pdf -O medical_ai_paper.pdf
!wget https://arxiv.org/pdf/2101.00027.pdf -O healthcare_ml_paper.pdf

--2026-03-06 09:27:05--  https://arxiv.org/pdf/2003.09871.pdf
Resolving arxiv.org (arxiv.org)... 151.101.3.42, 151.101.195.42, 151.101.131.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.3.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /pdf/2003.09871 [following]
--2026-03-06 09:27:05--  https://arxiv.org/pdf/2003.09871
Reusing existing connection to arxiv.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1439844 (1.4M) [application/pdf]
Saving to: ‘covid_paper.pdf’

covid_paper.pdf     100%[===================>]   1.37M  --.-KB/s    in 0.03s   

2026-03-06 09:27:05 (41.5 MB/s) - ‘covid_paper.pdf’ saved [1439844/1439844]

--2026-03-06 09:27:05--  https://arxiv.org/pdf/2005.00001.pdf
Resolving arxiv.org (arxiv.org)... 151.101.3.42, 151.101.195.42, 151.101.131.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.3.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /pdf/2005.00001 [f

In [14]:
# Verify the downloaded file
!ls -lh

total 3.5M
-rw-r--r-- 1 root root 1.4M Jan 23  2023 covid_paper.pdf
-rw-r--r-- 1 root root 1.5M Jan 23  2023 healthcare_ml_paper.pdf
-rw-r--r-- 1 root root 633K Jan 23  2023 medical_ai_paper.pdf
-rw-r--r-- 1 root root  176 Mar  6 09:24 Modelfile
drwxr-xr-x 1 root root 4.0K Feb  6 14:31 sample_data


In [15]:
# Load All PDFs Automatically
documents = []

pdf_files = [
    "covid_paper.pdf",
    "medical_ai_paper.pdf",
    "healthcare_ml_paper.pdf"
]

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print("Total pages loaded:", len(documents))

Total pages loaded: 67


In [16]:
# Split Documents into Chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 550


In [17]:
# Create Embeddings
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
# Create Vector Database
vector_db = Chroma.from_documents(
    chunks,
    embedding
)

In [19]:
# Connect Ollama Llama3
llm = Ollama(model="llama3")

retriever = vector_db.as_retriever(search_kwargs={"k":5})

# Build Multi-Document RAG Chatbot
medical_chatbot = RetrievalQA.from_chain_type(
    llm=llm,
    retriever = retriever
)

# Ask Questions
query = "How is chest radiography used to detect COVID-19?"

print(medical_chatbot.run(query))

According to the provided context, chest radiography imaging (e.g., chest X-ray (CXR) or computed tomography (CT) imaging) is conducted and analyzed by radiologists to look for visual indicators associated with SARS-CoV-2 viral infection.
